# Dataset Profiling Freedom in the World

**Notebook 01 of 08**

### Purpose

This is the first step of the pipeline: understand the raw dataset *before* we modify anything. We profile the structure of `data/raw/FH_FIW_WIDEF.csv` with its dimensions, identifiers, data types, missing values, duplicates and score ranges â€” and record what we find.

### Scope

This notebook is **read-only**: it makes no changes to the raw data and writes nothing to disk. The metadata JSON (`data/raw/FH_FIW.json`) is explored in notebook 02; here we work with the CSV alone.

## Data and Inputs

| Item | Location |
|---|---|
| Raw wide CSV | `data/raw/FH_FIW_WIDEF.csv` |

The CSV is a Data360 SDMX-style *wide* export: each row is one economy Ã— one indicator series, with one column per year (2013-2026).

**Question this notebook answers:** *What is the structure and quality of the raw dataset?*

**Method:** we answer one small question at a time â€” dimensions, column names, identifiers, types, missingness, duplicates, score ranges â€” and interpret each result before moving on.

## Setup: imports and the project root

Before importing our helpers from `src/`, we make sure the project root is on Python's search path. The cell below walks up from the current directory until it finds the folder that contains `data/raw/FH_FIW_WIDEF.csv`. This makes the notebook work no matter which folder Jupyter was launched from.

In [1]:
import sys
from pathlib import Path

# Walk up from the notebook until we find the project root (the folder
# containing data/raw/FH_FIW_WIDEF.csv). This makes the import below work
# no matter which directory Jupyter was launched from.
current = Path.cwd()
while not (current / 'data' / 'raw' / 'FH_FIW_WIDEF.csv').exists():
    current = current.parent
    if current == current.parent:
        raise RuntimeError('Could not find the project root.')

if str(current) not in sys.path:
    sys.path.insert(0, str(current))

import pandas as pd
import numpy as np

from src.data_loader import load_raw_data

print('pandas', pd.__version__)
print('numpy', np.__version__)

pandas 3.0.5
numpy 2.5.1


### Interpretation

The setup ran without errors: the project root was found, the reusable `load_raw_data()` helper from `src/data_loader.py` was imported, and the libraries are available. From here on, every step is a small question about the data.

## 1. Load the raw data

**Question:** how do we load the raw CSV into a DataFrame?

**Method:** call the reusable `load_raw_data()` helper and preview the first three rows.

In [2]:
df = load_raw_data()
df.head(3)

,STRUCTURE,STRUCTURE_ID,ACTION,FREQ,REF_AREA,INDICATOR,SEX,AGE,URBANISATION,UNIT_MEASURE,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
0,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,COD,FH_FIW_F3,_Z,_Z,_Z,0_TO_4,...,0,0,0,0,0,0,0,0,0,0
1,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,MYS,FH_FIW_F3,_Z,_Z,_Z,0_TO_4,...,1,1,2,2,2,2,2,2,2,2
2,datastructure,WB.DATA360:DS_DATA360(1.2),I,A,TZA,FH_FIW_F4,_Z,_Z,_Z,0_TO_4,...,2,2,1,1,1,1,1,1,1,1


### Interpretation

The first rows show the structure of the export: columns such as `STRUCTURE` and `ACTION` describe how Data360 packaged the file, `REF_AREA` identifies the economy, `INDICATOR` identifies the series, and the year columns at the end hold the scores. The row for Congo, Dem. Rep. Ã— `FH_FIW_F3` shows 0 in every year â€” the dataset itself records a zero score for that economyâ€“question combination; it is not missing data.

## 2. Dimensions

**Question:** how many rows and columns does the dataset have?

**Method:** the `shape` attribute gives both at once.

In [3]:
df.shape

(7880, 53)

### Interpretation

The dataset has **7880 rows** and **53 columns**. 7880 = 197 economies Ã— 40 indicators: the export is a complete grid, one row per economyâ€“indicator pair. 53 = 39 descriptor columns plus the 14 year columns. We verify the economy and indicator counts directly in the next steps.

## 3. Column names

**Question:** what are the 53 columns, and what role does each group play?

**Method:** list every column with its position in the file.

In [4]:
pd.DataFrame({'position': range(1, len(df.columns) + 1), 'column': df.columns})

,position,column
0,1,STRUCTURE
1,2,STRUCTURE_ID
2,3,ACTION
3,4,FREQ
4,5,REF_AREA
5,6,INDICATOR
6,7,SEX
7,8,AGE
8,9,URBANISATION
9,10,UNIT_MEASURE


### Interpretation

The columns split into three groups:

1. **Series descriptors** `STRUCTURE`, `FREQ`, `ACTION`, â€¦ describe the Data360 export itself (fixed values such as *Annual* frequency).
2. **Identifiers and labels** `REF_AREA`/`REF_AREA_LABEL` (economy), `INDICATOR`/`INDICATOR_LABEL` (indicator and its question text), `UNIT_MEASURE`/`UNIT_MEASURE_LABEL` (scale).
3. **Year columns**  `2013` to `2026`, holding the scores.

The `*_LABEL` columns are the human-readable twins of the code columns: codes are what we join on, labels are what we display.

## 4. Economy identifiers

**Question:** which columns identify economies, and how many economies are covered?

**Method:** count distinct values of `REF_AREA` (code) and `REF_AREA_LABEL` (name), then look at codes that differ from plain ISO-3166.

In [5]:
print('Distinct economy codes:', df['REF_AREA'].nunique())
print('Distinct economy names:', df['REF_AREA_LABEL'].nunique())
print()

# Some Data360 codes differ from plain ISO-3166 - see how they map to labels
special_codes = ['XKX', 'TWN', 'KOR', 'BHS', 'COD']
df[df['REF_AREA'].isin(special_codes)][['REF_AREA', 'REF_AREA_LABEL']].drop_duplicates()

Distinct economy codes: 197
Distinct economy names: 197



,REF_AREA,REF_AREA_LABEL
0,COD,"Congo, Dem. Rep."
6,KOR,"Korea, Rep."
8,XKX,Kosovo
11,BHS,"Bahamas, The"
26,TWN,"Taiwan, China"


### Interpretation

There are **197 distinct economy codes** (and 197 matching names): the dataset covers 197 economies as presented by Data360. `REF_AREA` codes are Data360 identifiers â€” most look like ISO-3166, but some do not: `XKX` is Kosovo, `TWN` is Taiwan, China, `KOR` is Korea, Rep. We therefore always resolve names through `REF_AREA_LABEL` (or the metadata JSON in notebook 02), never by assuming a label from the code.

## 5. Indicator identifiers

**Question:** which columns identify indicators, how many are there, and what scales do they use?

**Method:** count distinct `INDICATOR` values and list each code with its `UNIT_MEASURE` scale.

In [6]:
print('Distinct indicator codes:', df['INDICATOR'].nunique())
print()
df[['INDICATOR', 'UNIT_MEASURE', 'UNIT_MEASURE_LABEL']].drop_duplicates().sort_values('INDICATOR')

Distinct indicator codes: 40



,INDICATOR,UNIT_MEASURE,UNIT_MEASURE_LABEL
4152,FH_FIW_A,0_TO_12,0-12 scale
4146,FH_FIW_A1,0_TO_4,0-4 scale
4148,FH_FIW_A2,0_TO_4,0-4 scale
4166,FH_FIW_A3,0_TO_4,0-4 scale
4143,FH_FIW_ADD_A,0_TO_4,0-4 scale
4140,FH_FIW_ADD_Q,0_TO_4,0-4 scale
4136,FH_FIW_B,0_TO_16,0-16 scale
4153,FH_FIW_B1,0_TO_4,0-4 scale
4155,FH_FIW_B2,0_TO_4,0-4 scale
4173,FH_FIW_B3,0_TO_4,0-4 scale


### Interpretation

There are **40 distinct indicators**, all prefixed `FH_FIW_`. The `UNIT_MEASURE` column records the scale of each indicator â€” and the scales differ: individual questions are `0_TO_4`, category subtotals `0_TO_12`/`0_TO_16`, political rights `0_TO_40`, civil liberties `0_TO_60`, the overall score `0_TO_100`, plus two ratings on a `1_TO_7` scale and the categorical `FH_FIW_STATUS`. Scores on different scales are **not comparable** without normalizing.

## 6. Years covered

**Question:** what years are in the dataset, and how do the year cells arrive?

**Method:** list the year columns and inspect the dtype and values of one of them.

In [7]:
year_cols = [c for c in df.columns if str(c).isdigit()]
print('Year columns found:', len(year_cols))
print(year_cols)
print()
print('dtype of the 2013 column:', df['2013'].dtype)
print('Sample 2013 values:', df['2013'].head(5).tolist())

Year columns found: 14
['2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']

dtype of the 2013 column: str
Sample 2013 values: ['0', '1', '3', '2', '4']


### Interpretation

The dataset covers **2013-2026** (14 year columns). The year columns are read as **strings** (`str` dtype) â€” even though the values look like numbers. When we reshape to long format in notebook 03, the year must be converted to an integer. This is exactly why profiling matters: dtype surprises are best found before cleaning.

## 7. Data types

**Question:** what data types does the wide frame use overall?

**Method:** count the dtypes of all 53 columns.

In [8]:
df.dtypes.value_counts()

str      50
int64     3
Name: count, dtype: int64

### Interpretation

**50 columns are `str` and only 3 are `int64`** (`DECIMALS`, `TIME_FORMAT`, `UNIT_MULT`  small descriptor fields). Everything else, including all year columns, arrives as strings. There is nothing to fix here: this is simply how the wide export arrives, and cleaning (notebook 03) performs the conversions deliberately.

## 8. Missing data

**Question:** how much missing data exists, and where is it?

**Method:** count missing cells in the whole frame, per year column, and in the non-year columns.

In [9]:
print('Total missing cells in the whole frame:', df.isna().sum().sum())
print()
print('Missing values per year column:')
print(df[year_cols].isna().sum().to_string())
print()
non_year_missing = df.isna().sum()[~df.columns.isin(year_cols)]
print('Missing cells in non-year columns:', int(non_year_missing.sum()))

Total missing cells in the whole frame: 2164

Missing values per year column:
2013      0
2014      0
2015      0
2016      0
2017     40
2018    236
2019    236
2020    236
2021    236
2022    236
2023    236
2024    236
2025    236
2026    236

Missing cells in non-year columns: 0


### Interpretation

Missingness exists **only in the year columns**  all 39 descriptor columns are complete (0 missing). The pattern is striking: the early years are fully populated (0 missing in 2013-2016), then 40 cells are missing in 2017 and 236 cells in each year from 2018 to 2026  2164 missing cells in total. Per project policy we do **not** impute these: missing stays missing unless a documented analytical reason says otherwise.

## 9. Duplicates

**Question:** are there duplicate economyâ€“indicator records?

**Method:** count rows that share the same `REF_AREA` + `INDICATOR` pair.

In [10]:
duplicated_rows = df.duplicated(subset=['REF_AREA', 'INDICATOR']).sum()
print('Duplicate (economy, indicator) rows:', duplicated_rows)
print('Expected rows in a complete grid (197 x 40):', 197 * 40)

Duplicate (economy, indicator) rows: 0
Expected rows in a complete grid (197 x 40): 7880


### Interpretation

**Zero duplicate (economy, indicator) pairs.** Every economyâ€“indicator series appears exactly once, so the 7880 rows are the complete 197 Ã— 40 grid with no repeats. No deduplication will be needed when cleaning.

## 10. Score ranges

**Question:** what are the observed score ranges on each scale, and do they match the nominal `UNIT_MEASURE` bounds?

**Method:** coerce the year columns to numeric, then take the min and max per row and group by `UNIT_MEASURE`.

In [11]:
year_values = df[year_cols].apply(pd.to_numeric, errors='coerce')

ranges = df[['UNIT_MEASURE']].copy()
ranges['observed_min'] = year_values.min(axis=1)
ranges['observed_max'] = year_values.max(axis=1)
ranges.drop_duplicates().sort_values('UNIT_MEASURE')

,UNIT_MEASURE,observed_min,observed_max
23,0_TO_100,30.0,41.0
47,0_TO_100,55.0,65.0
77,0_TO_100,82.0,84.0
109,0_TO_100,10.0,18.0
110,0_TO_100,75.0,93.0
...,...,...,...
5840,1_TO_7,1.0,6.0
6350,1_TO_7,4.0,6.0
6546,1_TO_7,3.0,7.0
6918,1_TO_7,1.0,3.0


Most ranges sit inside their nominal bounds, but two things deserve a closer look: values below zero on the `0_TO_40` political-rights scale, and the direction of the `1_TO_7` ratings.

In [12]:
# 1. Out-of-range values: PR scores below the nominal 0-40 scale
pr = df[df['INDICATOR'] == 'FH_FIW_PR'].copy()
pr['lowest_year_score'] = year_values[df['INDICATOR'] == 'FH_FIW_PR'].min(axis=1)
pr[pr['lowest_year_score'] < 0][['REF_AREA_LABEL', 'lowest_year_score']]

,REF_AREA_LABEL,lowest_year_score
5047,Sudan,-4.0
6263,South Sudan,-4.0
6364,China,-2.0
6686,Myanmar,-2.0
7275,Syrian Arab Republic,-3.0


In [13]:
# 2. Rating direction: the 1-7 ratings are inverted (1 = most free)
ratings = df[df['INDICATOR'].isin(['FH_FIW_PR', 'FH_FIW_PR_RATING']) &
             df['REF_AREA_LABEL'].isin(['Finland', 'South Sudan'])]
ratings.pivot_table(index='REF_AREA_LABEL', columns='INDICATOR', values='2026', aggfunc='first')

INDICATOR,FH_FIW_PR,FH_FIW_PR_RATING
REF_AREA_LABEL,,
Finland,40,1
South Sudan,-4,7


### Interpretation

Two findings that will matter throughout the project:

1. **Out-of-range scores exist.** `FH_FIW_PR` is nominally `0_TO_40`, yet South Sudan and Sudan record **−4** in some years. `UNIT_MEASURE` is a nominal description, not a validation bound — the cleaning notebook must not silently drop or clamp such values.
2. **The 1–7 ratings are inverted.** Finland (PR = 40, most free) has rating 1; South Sudan (PR = −4, least free) has rating 7. On every 0–n scale higher means more freedom; on the ratings lower means more freedom. Any chart that mixes them must handle the direction explicitly.

## Summary and next question

### What we learned

- The dataset is a **complete grid**: 7880 rows = 197 economies × 40 indicators, covering **2013–2026** in 14 year columns.
- Identifiers: `REF_AREA` (Data360 codes — some non-ISO) and `INDICATOR` (`FH_FIW_*` codes with 8 different scales).
- Year values arrive as **strings**; missing values exist only in the year columns (from 2017 onward); there are **no duplicates**.
- Scores can fall outside their nominal scale (e.g. PR = −4), and the 1–7 ratings run in the **opposite direction** to every other scale.

### Next question

*What do the metadata JSON and the CSV labels tell us about the dataset's definitions?* — that is notebook 02, the metadata and data dictionary.